# Ekido: resume a checkpoint on this GPU

One leg of the cross-GPU test, run by hand. It isn't a connected worker: Colab's FAQ restricts distributed workers on free runtimes, so never run the agent here. Check the platform's terms before using it.

1. Choose a T4 runtime. Kaggle's P100 isn't expected to work with this torch build.
2. Upload `GPUChain_Team_Pack_2026-11-05.zip` and the `handover` folder, zipped, from the 3060.
3. Run every cell, and **save the notebook with its outputs**. Those outputs are the evidence.
4. Download `from_this_gpu.zip`, and take it back to the 3060 for the last leg.

Never put a wallet key or `.env` here.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv
!pip install -q "transformers==5.22.1" "peft==0.21.2" safetensors datasets numpy eth-account

In [ ]:
import os, zipfile
from pathlib import Path

REPO_ZIP = Path("GPUChain_Team_Pack_2026-11-05.zip")  # Change these two if the platform puts uploads elsewhere.
HANDOVER_ZIP = Path("handover.zip")

zipfile.ZipFile(REPO_ZIP).extractall("src")
zipfile.ZipFile(HANDOVER_ZIP).extractall("incoming")
repo = Path("src/GPUchain").resolve()
handover = next(p.parent for p in Path("incoming").rglob("spec.json")).resolve()
print("repo:", repo)
print("checkpoint:", handover)

import torch
assert torch.cuda.is_available(), "No GPU in this runtime. Switch it to a T4, because a CPU run proves nothing here."
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0), "torch", torch.__version__)

In [ ]:
# Refuses to overwrite earlier evidence. Micro-batch 8 keeps the logical batch at 32. Precision is chosen from this GPU.
!python {repo}/experiments/cross_gpu_resume.py resume --checkpoint {handover} --out from_this_gpu --steps 50 --micro-batch 8 --device cuda

In [ ]:
import json, shutil
evidence = json.loads(Path("from_this_gpu/evidence.json").read_text())
print(json.dumps(evidence, indent=2))
assert evidence["evidence_type"] == "REAL GPU RUN"
shutil.make_archive("from_this_gpu", "zip", "from_this_gpu")
print("Download from_this_gpu.zip, then save this notebook with its outputs.")